<a id='part2-3'></a>
## RAG檢索流程

![Flows](../static/images/RAG_flows.svg)

##### 連線設定

In [ ]:
import re
from collections import defaultdict
import weaviate
from openai import OpenAI
from weaviate.classes.query import Filter
from weaviate.classes.query import MetadataQuery

weaviate_client = weaviate.connect_to_local()
collection = weaviate_client.collections.get("FeedbackLoopDocuments")

LLM_BASE_URL = "http://127.0.0.1:11434/v1"
LLM_MODEL = "gemma4:26b"
llm_client = OpenAI(base_url=LLM_BASE_URL, api_key="ollama")

EMBEDDING_BASE_URL = "http://127.0.0.1:11434/v1"
EMBED_MODEL = "nvidia/Nemotron-3-Embed-1B-BF16"
embedding_client = OpenAI(base_url=EMBEDDING_BASE_URL, api_key="EMPTY")

print("Weaviate(v4) 連線成功：", weaviate_client.is_ready())

Weaviate(v4) 連線成功： True


##### collections功能編輯

In [2]:
# client.collections.delete("FeedbackLoopDocuments")
# print("已刪除collection")

collections = weaviate_client.collections.list_all()
print(collections)

{'FeedbackLoopDocuments': _CollectionConfigSimple(name='FeedbackLoopDocuments', description=None, generative_config=None, properties=[_Property(name='chunk_id', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, text_analyzer=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=None, vectorizer=None, vectorizer_configs={}), _Property(name='document_id', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, text_analyzer=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=None, vectorizer=None, vectorizer_configs={}), _Property(name='source_type', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, text_analyzer=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_co

##### 查詢collection的document_id

In [3]:
response = collection.query.fetch_objects(
    limit=5,
    return_properties=["document_id", "title", "source_type"]
)

documents = {}
for obj in response.objects:
    document_id = obj.properties.get("document_id")
    if document_id and document_id not in documents:
        documents[document_id] = {
            "title": obj.properties.get("title", ""),
            "source_type": obj.properties.get("source_type", "")}

for document_id, metadata in documents.items():
    print(f"document_id: {document_id}")
    print(f"title: {metadata['title']}")
    print(f"source_type: {metadata['source_type']}\n")

In [13]:
DOCUMENT_ID = "8cccc5484fe94dda8a946b6b1ded3dda"

##### 將問題拆解成3個問題

In [14]:
def generate_three_queries(original_question):
    prompt_template = f"""You are an AI language model assistant. Your task is to generate three 
    different versions of the given user question to retrieve relevant documents from a vector 
    database. By generating multiple perspectives on the user question, your goal is to help
    the user overcome some of the limitations of the distance-based similarity search. 
    Provide these alternative questions separated by newlines. 
    
    Original question: {original_question}"""


    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": prompt_template}
        ],
        temperature=0 
    )
    content = response.choices[0].message.content
    queries = [line.strip() for line in content.split("\n") if line.strip()]
    return queries

##### 使用hybrid或是near_vector模式選擇

In [15]:
SEARCH_MODE = "hybrid"  # "hybrid" 或 "near_vector"

def get_embedding(text: str) -> list[float]:
    response = embedding_client.embeddings.create(model=EMBED_MODEL,input=text.replace("\n", " "))
    return response.data[0].embedding

def retrieve_chunks(query: str, limit: int = 3) -> list[dict]:
    query_vector = get_embedding(query)
    common_args = {
        "target_vector": "default",
        "limit": limit,
        "filters": Filter.by_property("document_id").equal(DOCUMENT_ID),
        "return_properties": [
            "chunk_id", "document_id", "title", "content",
            "url", "page_number", "chunk_index"
        ]
    }

    if SEARCH_MODE == "hybrid":
        response = collection.query.hybrid(
            query=query,
            vector=query_vector,
            alpha=0.5,
            return_metadata=MetadataQuery(score=True),
            **common_args
        )
    elif SEARCH_MODE == "near_vector":
        response = collection.query.near_vector(
            near_vector=query_vector,
            return_metadata=MetadataQuery(distance=True),
            **common_args
        )

    results = []
    for rank, obj in enumerate(response.objects, start=1):
        results.append({
            "uuid": str(obj.uuid),
            "rank": rank,
            "retrieval_score": getattr(obj.metadata, "score", None),
            "distance": getattr(obj.metadata, "distance", None),
            **obj.properties
        })
    return results

##### RRF向量排名(會有9個答案，進行排名取最高3個)

In [16]:
def reciprocal_rank_fusion(result_groups: list[list[dict]], k: int = 60) -> list[dict]:
    scores = defaultdict(float)
    documents = {}
    found_by = defaultdict(list)

    for query_no, docs in enumerate(result_groups, start=1):
        for doc in docs:
            doc_id = doc["uuid"]
            rank = doc["rank"]
            scores[doc_id] += 1 / (k + rank)
            documents[doc_id] = doc
            found_by[doc_id].append(f"問法 {query_no}（第 {rank} 名）")

    fused_docs = []
    for doc_id, score in scores.items():
        fused_docs.append({
            **documents[doc_id],
            "rrf_score": score,
            "found_by": found_by[doc_id],
        })

    return sorted(fused_docs, key=lambda doc: doc["rrf_score"], reverse=True)


def build_context(top_docs: list[dict]) -> str:
    references = []

    for index, doc in enumerate(top_docs, start=1):
        references.append(f"""[參考資料 {index}]
                        標題：{doc.get('title', '未命名')}
                        文件 ID：{doc.get('document_id', '')}
                        段落 ID：{doc.get('chunk_id', '')}
                        頁碼：{doc.get('page_number', '')}
                        內容：{doc.get('content', '')}""")
    return "\n\n".join(references)

In [17]:
question = "這份文件的主要內容與重要結論是什麼？"
generated_queries = generate_three_queries(question)[:3]

result_groups = []
for query_no, query in enumerate(generated_queries, start=1):
    docs = retrieve_chunks(query, limit=3)
    result_groups.append(docs)
    print(f"\nQ {query_no}:{query}")

    for doc in docs:
        metric = (
            f"score={doc['retrieval_score']}"
            if doc.get("retrieval_score") is not None
            else f"distance={doc['distance']}")
        print(f"Rank {doc['rank']} | {metric} | chunk_id={doc.get('chunk_id', '')}")

candidate_count = sum(len(docs) for docs in result_groups)
print(f"\nchunks：{candidate_count}")


fused_docs = reciprocal_rank_fusion(result_groups, k=60)
top3 = fused_docs[:3]

for index, doc in enumerate(top3, start=1):
    print(f"\nTop {index} | RRF={doc['rrf_score']:.6f}")
    print(f"命中來源：{'、'.join(doc['found_by'])}")
    print(f"chunk_id：{doc.get('chunk_id', '')}")
    print(doc.get('content', '')[:300])



Q 1:請摘要這份文件的核心要點與關鍵發現。
Rank 1 | score=0.5 | chunk_id=chunk_00002
Rank 2 | score=0.3262251019477844 | chunk_id=chunk_00001
Rank 3 | score=0.0 | chunk_id=chunk_00003

Q 2:這份文件的核心主題是什麼，以及它得出了哪些重要的研究成果？
Rank 1 | score=0.5 | chunk_id=chunk_00002
Rank 2 | score=0.30779075622558594 | chunk_id=chunk_00003
Rank 3 | score=0.0 | chunk_id=chunk_00001

Q 3:請提供此文件的內容大綱以及其最終的總結性結論。
Rank 1 | score=0.5 | chunk_id=chunk_00003
Rank 2 | score=0.014692317694425583 | chunk_id=chunk_00002
Rank 3 | score=0.0 | chunk_id=chunk_00001

chunks：9

Top 1 | RRF=0.048916
命中來源：問法 1（第 1 名）、問法 2（第 1 名）、問法 3（第 2 名）
chunk_id：chunk_00002
C. 獲選口頭報告作品將於115 年10 月05 日前接到通知，請準備15

分鐘之口頭報告於癌登年會當日(10 月24 日)分享。

D. 所有入圍作品皆須以紙本海報的形式於研討會當日張貼，研討會

將安排討論時間，供其他與會者與海報作者提問交流（第一作者

若無法出席研討會，請協調共同作者口頭報告或張貼海報）。

7. 評審辦法：由審稿委員負責評分，評分重點為：

A. 研究方法與發現佔80%

B. 海報設計與呈現20%

8. 比賽結果除口頭報告者會提前告知外，其餘獎項結果將於研討會當日

公布。

9. 活動獎勵：

獎項 獎勵 特優 獎金10000 元+獎狀1 張 優等 獎金600

Top 2 | RRF=0.048395
命中來源：問法 1（第 3 名）、問法 2（第 2 名）、問法 3（第 1 名）
chunk_id：chunk_00003
F. 海報規格

In [18]:
context = build_context(top3)
answer_response = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {
            "role": "system",
            "content": """你是嚴謹的文件問答助手，只能根據提供的參考資料回答，不得編造內容，若資料不足，請回答「提供的參考資料不足以回答此問題」。\n
                        請用繁體中文回答，並在最後列出實際使用的參考資料編號。""",
        },
        {
            "role": "user",
            "content": f"""問題：{question}
                    參考資料：{context}
                    請回答問題。""",
        },
    ],
    temperature=0.1
)

final_answer = answer_response.choices[0].message.content.strip()
print(f"LLM回答:{final_answer}")

LLM回答:這份文件的主要內容是關於「2026 年台灣癌症登記學會學術海報發表比賽」的辦法，詳細說明了比賽的宗旨、參加方式、評審標準及獎勵機制。

**主要內容：**
1.  **活動目的與時間**：此比賽旨在提供癌登師交流學習的平台。115 年的學術研討會將於 10 月 24 日在國立成功大學醫學院舉行（實體會議）。
2.  **參加對象與投稿資訊**：
    *   對象：本會正式會員（含永久、個人、學生、團體會員）。
    *   投稿時間：即日起至民國 115 年 06 月 30 日止。
    *   投稿規格：每篇 1000 字內的研究摘要，可包含至多兩個圖或表。
    *   投稿方式：透過電子郵件（tscr.doc@gmail.com）投稿。
    *   比賽分組：分為「癌登相關學術分析」及「癌決相關行政事務（如創意年報、作業改善、教育）」兩大組別。
3.  **評審與流程**：
    *   評分重點：研究方法與發現佔 80%，海報設計與呈現佔 20%。
    *   入圍通知：入圍結果於 115 年 8 月 03 日公布；獲選口頭報告者將於 10 月 05 日前接到通知。
    *   入圍後義務：入圍作品須以紙本海報形式於研討會當日張貼，並安排討論時間；入圍者須於 115 年 8 月 31 日前寄送電子海報（PDF 檔）進行複審。
4.  **獎勵與規範**：
    *   獎項包含特優、優等、佳作、入圍，並提供不同金額的獎金與獎狀。
    *   若研究內容已於 114 年 10 月 18 日前發表，則不進入獲獎名單，僅以入圍海報形式分享。
    *   投稿者須遵守著作權法，且若稿件內容不符規定（如篇幅過長、主題不一、與癌症登記無直接相關或論文發表超過 2 年），審稿委員得要求退件。

**重要結論：**
*   比賽旨在促進癌登領域工作夥伴的研究成果分享。
*   評審核心在於「研究方法與發現」。
*   入圍者需配合於研討會當日進行海報展示與交流。

參考資料編號：[參考資料 1], [參考資料 2], [參考資料 3]
